In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Data Reading

In [2]:
df = pd.read_csv(
    '../data/wustl_iiot_2021.csv',
    parse_dates=['StartTime', 'LastTime']
)

### Feature Engineering

In [3]:
# none for now, requires more research into the types of attacks to create useful features

### Conversion of dataset into a format suitable for sequence modelling

In [4]:
# length of sequence prior to classify the next traffic type
SEQUENCE_LENGTH = 5

In [5]:
df = (
    df.sort_values(by=['SrcAddr', 'StartTime', 'LastTime'])
        .reset_index(drop=True)
)

df.head()

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,SAppBytes,DAppBytes,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target
0,2019-08-19 09:46:08,2019-08-19 14:14:18,0,0,0,0,0,0,0,0,...,0,0,0,0.0,0.0,0,0.0,0.0,normal,0
1,2019-08-19 09:46:18,2019-08-19 09:45:18,0,5093,0,0,2,52380,2845,52380,...,5093,0,1624118724,0.0,0.0,0,0.0,0.0,normal,0
2,2019-08-19 09:47:18,2019-08-19 09:46:18,0,5087,0,0,2,52407,2847,52407,...,5087,0,1628918892,0.0,0.0,0,0.0,0.0,normal,0
3,2019-08-19 09:48:18,2019-08-19 09:47:18,0,5085,0,0,2,52419,2851,52419,...,5085,0,1633708836,0.0,0.0,0,0.0,0.0,normal,0
4,2019-08-19 09:49:18,2019-08-19 09:48:18,0,5096,0,0,2,52464,2858,52464,...,5096,0,1638519228,0.0,0.0,0,0.0,0.0,normal,0


In [6]:
exclude_cols = ['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'sIpId', 'dIpId', 'Traffic', 'Target']
feature_cols = [c for c in df.columns if c not in exclude_cols]

In [7]:
# lookback length
X_list = []
target_list = []
traffic_list = []

for src_addr, group in df.groupby('SrcAddr'):
    group = group.sort_values('StartTime').reset_index(drop=True)
    feats = group[feature_cols].values
    targets = group['Target'].values
    traffics = group['Traffic'].values
    n = len(group)

    if n <= SEQUENCE_LENGTH:
        # not enough history to form even one window + target
        # which shouldn't happen since minimum number of rows for source ip is 8
        # so raise error if this occurs
        raise ValueError('Not enough history to form one window + target')

    for i in range(0, n - SEQUENCE_LENGTH, 1):
        window = feats[i : i + SEQUENCE_LENGTH]
        target = targets[i + SEQUENCE_LENGTH]
        traffic = traffics[i + SEQUENCE_LENGTH]

        X_list.append(window)
        target_list.append(target)
        traffic_list.append(traffic)

X = np.array(X_list)
y_traffic = np.array(traffic_list)
y_target = np.array(target_list)

In [8]:
X.shape

(1194394, 5, 41)

In [9]:
y_traffic.shape

(1194394,)

In [10]:
y_target.shape

(1194394,)

Now, we have a sequence of lookback of a window of 5, to predict the next traffic class